# SvelteNeck-YOLO26 & SvelteF3M — Complete Ablation Pipeline (v8)
Dataset: `mexwell/trashcan-1-0` (Automatic fast in-memory conversion of COCO JSON -> YOLO)

Stages:
- **Stage 1A**: YOLO26n Baseline (50 epochs)
- **Stage 1B**: YOLO11n Reference (50 epochs)
- **Stage 2A**: SvelteNeck-YOLO26 (e=0.50, Lightweight Target)
- **Stage 2B**: SvelteNeck-YOLO26 (e=0.75, Higher Capacity)
- **Stage 3B**: SvelteF3M-YOLO26 (SvelteNeck + F3M Attention Gates, PRIMARY MODEL)

In [ ]:
!nvidia-smi
!pip install -q 'ultralytics>=8.3.0' --upgrade
import torch
print(f'PyTorch {torch.__version__}, CUDA {torch.version.cuda}, GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
import os, json, yaml, shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

print('Converting TrashCan-Instance COCO -> YOLO in /kaggle/working/data ...')

# 1. Locate instances_train_trashcan.json and instances_val_trashcan.json
train_json, val_json = None, None
for r, d, files in os.walk('/kaggle/input'):
    if 'instances_train_trashcan.json' in files and 'instance_version' in r:
        train_json = Path(r) / 'instances_train_trashcan.json'
        val_json   = Path(r) / 'instances_val_trashcan.json'
        train_img_src = Path(r) / 'train'
        val_img_src   = Path(r) / 'val'
        break

assert train_json is not None and train_json.exists(), 'Could not find instance_version JSONs'
print(f'Source JSON: {train_json.parent}')

OUT = Path('/kaggle/working/trashcan_yolo')
for split in ['train', 'val']:
    (OUT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUT / 'labels' / split).mkdir(parents=True, exist_ok=True)

with open(train_json, 'r') as f:
    t_data = json.load(f)

# Categories mapping
cats = sorted(t_data['categories'], key=lambda c: c['id'])
cat_map = {c['id']: idx for idx, c in enumerate(cats)}
NC = len(cats)
NAMES = [c['name'] for c in cats]
print(f'Detected {NC} classes: {NAMES[:4]} ...')

def convert_split(json_path, img_src_dir, split_name):
    with open(json_path, 'r') as f:
        data = json.load(f)
    img_meta = {img['id']: img for img in data['images']}
    anns_by_img = {}
    for ann in data['annotations']:
        anns_by_img.setdefault(ann['image_id'], []).append(ann)
    
    def process_img(img_id, info):
        fn = info['file_name']
        w, h = info['width'], info['height']
        src_file = img_src_dir / fn
        dst_img = OUT / 'images' / split_name / fn
        if not src_file.exists(): return
        # Symlink for ultra-fast zero-disk-copy
        if not dst_img.exists():
            os.symlink(src_file, dst_img)
        
        txt_lines = []
        for ann in anns_by_img.get(img_id, []):
            cid = ann.get('category_id')
            if cid not in cat_map: continue
            cls_idx = cat_map[cid]
            bbox = ann.get('bbox', [])
            if len(bbox) != 4: continue
            x, y, bw, bh = bbox
            # Clip boundaries
            x1, y1 = max(0.0, min(float(x), float(w))), max(0.0, min(float(y), float(h)))
            x2, y2 = max(0.0, min(float(x + bw), float(w))), max(0.0, min(float(y + bh), float(h)))
            cw, ch = x2 - x1, y2 - y1
            if cw <= 0 or ch <= 0: continue
            xc, yc = (x1 + cw / 2.0) / w, (y1 + ch / 2.0) / h
            nw, nh = cw / w, ch / h
            txt_lines.append(f'{cls_idx} {xc:.6f} {yc:.6f} {nw:.6f} {nh:.6f}')
        
        dst_lbl = OUT / 'labels' / split_name / (Path(fn).stem + '.txt')
        with open(dst_lbl, 'w') as lf:
            lf.write('\n'.join(txt_lines) + '\n')

    with ThreadPoolExecutor(max_workers=8) as ex:
        ex.map(lambda item: process_img(item[0], item[1]), img_meta.items())
    print(f'Finished {split_name} split conversion.')

convert_split(train_json, train_img_src, 'train')
convert_split(val_json, val_img_src, 'val')

DATA_YAML = '/kaggle/working/trashcan_instance.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump({'path': str(OUT), 'train': 'images/train', 'val': 'images/val', 'nc': NC, 'names': NAMES}, f)
print('Dataset YAML ready:')
print(open(DATA_YAML).read())
print(f'Train images created: {len(list((OUT/"images"/"train").glob("*.jpg")))}')
print(f'Train labels created: {len(list((OUT/"labels"/"train").glob("*.txt")))}')

In [ ]:
HYP = dict(data=DATA_YAML, imgsz=640, epochs=50, optimizer='AdamW',
           lr0=0.001, lrf=0.01, momentum=0.937, weight_decay=0.0005,
           warmup_epochs=3, seed=42, deterministic=True, device=0,
           workers=4, amp=True, cache=False, val=True, save=True, plots=True,
           project='/kaggle/working/runs',
           hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, degrees=5.0, translate=0.1,
           scale=0.5, mosaic=1.0, mixup=0.15, copy_paste=0.1, fliplr=0.5)
print('Hyperparameters configured.')

In [ ]:
from ultralytics import YOLO
print('========================================')
print('STAGE 1A: Training YOLO26n Baseline')
print('========================================')
YOLO('yolo26n.pt').train(**HYP, batch=16, name='s1a_yolo26n')
CKPT_1A = '/kaggle/working/runs/s1a_yolo26n/weights/best.pt'
v1a = YOLO(CKPT_1A).val(data=DATA_YAML, imgsz=640, batch=16, verbose=False)
print(f'Stage 1A Results: mAP50={v1a.box.map50:.4f}, mAP50-95={v1a.box.map:.4f}')

In [ ]:
from ultralytics import YOLO
print('========================================')
print('STAGE 1B: Training YOLO11n Reference')
print('========================================')
YOLO('yolo11n.pt').train(**HYP, batch=16, name='s1b_yolo11n')
CKPT_1B = '/kaggle/working/runs/s1b_yolo11n/weights/best.pt'
v1b = YOLO(CKPT_1B).val(data=DATA_YAML, imgsz=640, batch=16, verbose=False)
print(f'Stage 1B Results: mAP50={v1b.box.map50:.4f}, mAP50-95={v1b.box.map:.4f}')

In [ ]:
import ultralytics, importlib
from pathlib import Path

NN = Path(ultralytics.__file__).parent / 'nn' / 'modules'

CEH_CODE = '''
import torch
import torch.nn as nn

class EHConv(nn.Module):
    def __init__(self, c1, c2, k=3, s=1):
        super().__init__()
        p = k // 2
        self.dw = nn.Conv2d(c1, c1, k, s, p, groups=c1, bias=False)
        self.pw = nn.Conv2d(c1, c2, 1, bias=False)
        self.bn = nn.BatchNorm2d(c2)
        self.act = nn.Mish()
    def forward(self, x):
        return self.act(self.bn(self.pw(self.dw(x))))

class EHSBottleneck(nn.Module):
    def __init__(self, c1, c2, shortcut=True, e=1.0, k=5, s=1):
        super().__init__()
        h = int(c2 * e)
        self.cv1 = EHConv(c1, h, k=1)
        self.cv2 = EHConv(h, c2, k=k, s=s)
        self.add = shortcut and c1 == c2 and s == 1
    def forward(self, x):
        return x + self.cv2(self.cv1(x)) if self.add else self.cv2(self.cv1(x))

class EHSCSP(nn.Module):
    def __init__(self, c1, c2, n=1, shortcut=True, e=1.0, k=5):
        super().__init__()
        h = int(c2 * e)
        self.cv1 = nn.Sequential(nn.Conv2d(c1, 2*h, 1, bias=False), nn.BatchNorm2d(2*h), nn.Mish())
        self.blocks = nn.ModuleList([EHSBottleneck(h, h, shortcut=shortcut, e=1., k=k) for _ in range(n)])
        self.cv2 = nn.Sequential(nn.Conv2d((n+2)*h, c2, 1, bias=False), nn.BatchNorm2d(c2), nn.Mish())
    def forward(self, x):
        x = self.cv1(x)
        p, t = torch.chunk(x, 2, dim=1)
        s = [p, t]
        for b in self.blocks:
            t = b(t)
            s.append(t)
        return self.cv2(torch.cat(s, dim=1))

class SvelteNeck(nn.Module):
    def __init__(self, e=0.50):
        super().__init__()
        self.upsample_p5 = nn.Upsample(scale_factor=2, mode="nearest")
        self.td_p4 = EHSCSP(384, 128, n=1, shortcut=True, e=e, k=5)
        self.upsample_p4 = nn.Upsample(scale_factor=2, mode="nearest")
        self.td_p3 = EHSCSP(256, 64,  n=1, shortcut=True, e=e, k=5)
        self.bu_p3_down = EHConv(64,  64,  k=3, s=2)
        self.bu_p4 = EHSCSP(192, 128, n=1, shortcut=True, e=e, k=5)
        self.bu_p4_down = EHConv(128, 128, k=3, s=2)
        self.bu_p5 = EHSCSP(384, 256, n=1, shortcut=True, e=e, k=5)
    def forward(self, F):
        p3, p4, p5 = F
        u5 = self.upsample_p5(p5)
        t4 = self.td_p4(torch.cat([u5, p4], 1))
        u4 = self.upsample_p4(t4)
        t3 = self.td_p3(torch.cat([u4, p3], 1))
        d3 = self.bu_p3_down(t3)
        b4 = self.bu_p4(torch.cat([d3, t4], 1))
        d4 = self.bu_p4_down(b4)
        b5 = self.bu_p5(torch.cat([d4, p5], 1))
        return [t3, b4, b5]

class F3M(nn.Module):
    def __init__(self, c, r=4, sk=7):
        super().__init__()
        rd = max(1, c // r)
        self.cg = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(c, rd, bias=False), nn.ReLU(inplace=True),
            nn.Linear(rd, c, bias=False), nn.Sigmoid())
        self.sg = nn.Sequential(
            nn.Conv2d(c, c, sk, 1, sk//2, groups=c, bias=False),
            nn.BatchNorm2d(c), nn.Sigmoid())
    def forward(self, x):
        cg = self.cg(x).view(x.shape[0], x.shape[1], 1, 1)
        x  = x * cg
        return x * self.sg(x)

class Index(nn.Module):
    def __init__(self, c, idx=0):
        super().__init__()
        self.idx = idx
    def forward(self, x):
        return x[self.idx]
'''
(NN / 'ceh.py').write_text(CEH_CODE)

init_p = NN / '__init__.py'; t = init_p.read_text()
if 'SvelteNeck' not in t:
    init_p.write_text(t + '\nfrom .ceh import EHConv, EHSBottleneck, EHSCSP, SvelteNeck, F3M, Index\n')

tasks_p = Path(ultralytics.__file__).parent / 'nn' / 'tasks.py'; tt = tasks_p.read_text()
if 'SvelteNeck' not in tt:
    patch = '\ntry:\n    from ultralytics.nn.modules.ceh import EHConv, EHSBottleneck, EHSCSP, SvelteNeck, F3M, Index\nexcept ImportError:\n    pass\n'
    tasks_p.write_text(patch + tt)

importlib.reload(ultralytics.nn.modules)
from ultralytics.nn.modules.ceh import SvelteNeck, F3M, Index
print('Ultralytics custom modules successfully registered:', SvelteNeck, F3M, Index)

In [ ]:
BB = (
    f'nc: {NC}\n'
    'end2end: true\n'
    'reg_max: 1\n'
    'scales:\n'
    '  n: [0.50, 0.25, 1024]\n'
    'scale: n\n'
    'backbone:\n'
    '  - [-1, 1, Conv,  [64,  3, 2]]\n'
    '  - [-1, 1, Conv,  [128, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [256, False, 0.25]]\n'
    '  - [-1, 1, Conv,  [256, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [512, False, 0.25]]\n'
    '  - [-1, 1, Conv,  [512, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [512, True]]\n'
    '  - [-1, 1, Conv,  [1024, 3, 2]]\n'
    '  - [-1, 2, C3k2,  [1024, True]]\n'
    '  - [-1, 1, SPPF,  [1024, 5, 3, True]]\n'
    '  - [-1, 2, C2PSA, [1024]]\n'
)

# Config Stage 2A: e=0.50 (low FLOP budget target)
Path('/kaggle/working/svn050.yaml').write_text(
    BB
    + 'head:\n'
    + '  - [[4, 6, 10], 1, SvelteNeck, [0.50]]\n'
    + '  - [11, 1, Index, [64, 0]]\n'
    + '  - [11, 1, Index, [128, 1]]\n'
    + '  - [11, 1, Index, [256, 2]]\n'
    + '  - [[12, 13, 14], 1, Detect, [nc]]\n'
)
# Config Stage 2B: e=0.75
Path('/kaggle/working/svn075.yaml').write_text(
    BB
    + 'head:\n'
    + '  - [[4, 6, 10], 1, SvelteNeck, [0.75]]\n'
    + '  - [11, 1, Index, [64, 0]]\n'
    + '  - [11, 1, Index, [128, 1]]\n'
    + '  - [11, 1, Index, [256, 2]]\n'
    + '  - [[12, 13, 14], 1, Detect, [nc]]\n'
)
# Config Stage 3B: SvelteNeck + F3M Attention Gates
Path('/kaggle/working/svnf3m.yaml').write_text(
    BB
    + 'head:\n'
    + '  - [[4, 6, 10], 1, SvelteNeck, [0.50]]\n'
    + '  - [11, 1, Index, [64, 0]]\n'
    + '  - [11, 1, Index, [128, 1]]\n'
    + '  - [11, 1, Index, [256, 2]]\n'
    + '  - [12, 1, F3M, [64]]\n'
    + '  - [13, 1, F3M, [128]]\n'
    + '  - [14, 1, F3M, [256]]\n'
    + '  - [[15, 16, 17], 1, Detect, [nc]]\n'
)
print('All model YAML definitions written.')

In [ ]:
import torch
from ultralytics import YOLO

def xfer(src_path, dst_model, n_layers=10):
    ck = torch.load(src_path, map_location='cpu', weights_only=False)
    src_m = ck.get('model', None)
    if src_m is None: return dst_model
    sd = src_m.state_dict() if hasattr(src_m, 'state_dict') else src_m
    dd = dst_model.model.state_dict()
    tr = {k: v for k, v in sd.items()
          if k.startswith('model.') and k.split('.')[1].isdigit()
          and int(k.split('.')[1]) <= n_layers
          and k in dd and dd[k].shape == v.shape}
    dd.update(tr)
    dst_model.model.load_state_dict(dd, strict=False)
    print(f'Transferred {len(tr)} backbone weights from {src_path}')
    return dst_model

print('Weight transfer function ready.')

In [ ]:
from ultralytics import YOLO
print('==============================================')
print('STAGE 2A: SvelteNeck-YOLO26 (e=0.50) Training')
print('==============================================')
m2a = xfer(CKPT_1A, YOLO('/kaggle/working/svn050.yaml'))
m2a.train(**{**HYP, 'batch': 8, 'lr0': 5e-4, 'name': 's2a_svn050'})
CKPT_2A = '/kaggle/working/runs/s2a_svn050/weights/best.pt'
v2a = YOLO(CKPT_2A).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
print(f'Stage 2A Results: mAP50={v2a.box.map50:.4f}, mAP50-95={v2a.box.map:.4f}')

In [ ]:
from ultralytics import YOLO
print('==============================================')
print('STAGE 2B: SvelteNeck-YOLO26 (e=0.75) Training')
print('==============================================')
m2b = xfer(CKPT_1A, YOLO('/kaggle/working/svn075.yaml'))
m2b.train(**{**HYP, 'batch': 8, 'lr0': 5e-4, 'name': 's2b_svn075'})
CKPT_2B = '/kaggle/working/runs/s2b_svn075/weights/best.pt'
v2b = YOLO(CKPT_2B).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
print(f'Stage 2B Results: mAP50={v2b.box.map50:.4f}, mAP50-95={v2b.box.map:.4f}')

In [ ]:
from ultralytics import YOLO
print('===========================================================')
print('STAGE 3B: SvelteF3M-YOLO26 (SvelteNeck + F3M PRIMARY MODEL)')
print('===========================================================')
m3b = xfer(CKPT_1A, YOLO('/kaggle/working/svnf3m.yaml'))
m3b.train(**{**HYP, 'batch': 8, 'lr0': 5e-4, 'name': 's3b_svnf3m'})
CKPT_3B = '/kaggle/working/runs/s3b_svnf3m/weights/best.pt'
v3b = YOLO(CKPT_3B).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
print(f'Stage 3B Results: mAP50={v3b.box.map50:.4f}, mAP50-95={v3b.box.map:.4f}')

In [ ]:
import csv
from pathlib import Path
from ultralytics import YOLO

ckpts = {
    'YOLO26n Baseline': CKPT_1A,
    'YOLO11n Reference': CKPT_1B,
    'SvelteNeck-e050':   CKPT_2A,
    'SvelteNeck-e075':   CKPT_2B,
    'SvelteF3M-YOLO26 (Primary)': CKPT_3B,
}

rows = []
for name, ck in ckpts.items():
    if not Path(ck).exists(): continue
    v    = YOLO(ck).val(data=DATA_YAML, imgsz=640, batch=8, verbose=False)
    info = YOLO(ck).info(verbose=False)
    pm   = round(info[1]/1e6, 3) if isinstance(info, (list, tuple)) else float('nan')
    gf   = round(info[3], 2)     if isinstance(info, (list, tuple)) else float('nan')
    row  = dict(
        model=name,
        mAP50=round(float(v.box.map50), 4),
        mAP5095=round(float(v.box.map), 4),
        precision=round(float(v.box.mp), 4),
        recall=round(float(v.box.mr), 4),
        params_M=pm,
        GFLOPs=gf,
        checkpoint=ck
    )
    rows.append(row)
    print(f"{name:<28} | mAP50: {row['mAP50']:.4f} | mAP50-95: {row['mAP5095']:.4f} | Params: {pm}M | GFLOPs: {gf}")

if rows:
    with open('/kaggle/working/ablation_results.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        w.writeheader()
        w.writerows(rows)
    print('\nSaved /kaggle/working/ablation_results.csv')

print('\nAll Trained Weights:')
for ck in sorted(Path('/kaggle/working/runs').rglob('best.pt')):
    print(f'  {ck}  ({ck.stat().st_size / 1e6:.1f} MB)')